# RealLabel Tutorial

A guide to running RealLabel via `checkmaite`.

## What is RealLabel?

RealLabel is a **quality assurance tool** designed to **identify missing or erroneous ground-truth labels** in object-detection datasets.

RealLabel labels images in a labeled dataset as:
- Likely Correct (LCL) (True Positive)
- Likely Wrong (LWL) (False Positive)
- Likely Missed (LML) (False Negative)
- Mis-Classification (LWL + LML)

For example, assuming you start with the 3 labeled (black boxes) objects in the dataset below (ignore green boxes for now) . . .

![label definitions](../assets/reallabel-original-labels.png)

Reallabel will produce **LCL, LWL, LML, or LWL+LML labels** such as

![](../assets/reallabel-assignments.png)

![](../assets/reallabel-LCL-LWL.png)

![](../assets/reallabel-LML-MC.png)

## How does RealLabel work?

At high level, the RealLabel algorithm consists of 4 steps:
1. Run multiple object detection models on a single labeled dataset
2. Identify groups of bounding boxes
3. Calculate the Aggregated Condfidence Score on each box group
4. Use the AC Score to classify the groups with RealLabel labels

Each of these steps are discussed in more detail below.

### 1. Run multiple object detection models on a labeled object detection dataset

The labeled dateset (ground truth) consists of:
1. **bounding box coordinates** e.g. ((x1, y1), (x2, y2))
2. **labels** e.g. person, truck, etc.

for each object labelled in a set of images.  

Running object detection (OD) models on the labeled dataset also produces **bounding box coordinates** and **labels** as well as a **confidence score** ranging from 0-1 for identified objects in an image. RealLabel relies on consensus (or lack thereof) among multiple OD models to assess the quality of the ground truth labels. The resulting quality assessment (e.g. Likely Correct, Likely Wrong, Likely Missed, Mis-Classification) for the ground truth labels are called **RealLabel assignments**.

<div class="admonition warning"> 
    <p class="admonition-title">Model Quality Impact</p>
    <p><b>Warning:</b> The quality and accuracy of RealLabel assignments are directly dependent on the performance of the object detection models used as input. Since RealLabel processes the models' inferences (detections and their associated bounding boxes and confidences) to arrive at its results, models with poor detection performance will provide unreliable data, inevitably resulting in inaccurate RealLabel assignments.</p>
</div>

<div class="admonition info"> 
    <p class="admonition-title">Unlabeled Dataset Usage</p>
    <p><b>Note:</b> It's also possible to run RealLabel to take a first pass at labelling an unlabeled image dataset. For this use case, set <b>run_with_ground_truth=False</b> in the config (described below).  Additionally, the only RealLabel assignment that will be made is "Likely Missing Label" (False Negative) due to the lack of labels.</p>
</div>

### 2. Group the boxes

After running the OD models on the dataset, RealLabel will group the boxes.  RealLabel provides the best results if all models are skilled at detecting each of the objects in the labeled dataset. This results in "groups" of colocated bounding boxes as seen in the example below. The 3 green boxes represent 3 OD models detecting the butterfly and the same 3 models also detecting the flower.

![](../assets/reallabel-labeled-img-ex.png)

Reallabel formally groups bounding boxes into "box groups" using one of two currently supported **deduplication algorithms**, and generates a **box group winner** box for each group of bounding boxes.  This group winner box will be returned in the results dataframe.  

The two algorithms for grouping bounding boxes are **Non Maximum Suppression (NMS)** and **Weighted-Box Fusion (WBF)**. 
- **Non Maximum Suppression** chooses the bounding box with the highest confidence as the best representative (more details below).
- **Weighted-Box Fusion** combines bounding boxes from multiple models without discarding any bounding box. it uses confidence scores of all the bounding boxes to construct the averaged bounding box. 

The checkmaite UI utilizes the NMS algorithm which is the default in RealLabel.

![](../assets/reallabel-box-group-winners.png)

#### Non Maximum Suppression
**Non Maximum Suppression (NMS)** orders all object detections by model confidence. The bounding box detected with most confidence is used to calculate a pairwise **Intersection Over Union (IoU) score** with all remaining bounding boxes.  IoU score is the intersection area of 2 bounding boxes divided by the union area of the same 2 bounding boxes. The high confidence bounding box and all boxes with IoU score above a threshold are removed from the list of bounding boxes and considered to be a single box group. The process is repeated until all bounding boxes are assigned a group.

#### Calibrated Scores
Model predictions are not always well calibrated - for instance, a model might consistently produce overly high confidence scores for all bounding boxes.  Reallabel can adjust these scores using known ground truth data.

### 3. Calculate Aggregated Confidence (AC)

**Aggregated Confidence (AC)** is calculated for each group by summing confidence scores from all models and dividing by number of models that detected an object in that group.

### 4. Classify some box groups with RealLabel assignments
- **Likely Correct** (True Positive): Ground truth exists AND AC > threshold
- **Likely Wrong** (False Positive): Ground truth exists BUT AC ≤ threshold
- **Likely Missed** (False Negative): No ground truth BUT AC ≥ threshold
- **Mis-classified**: Ground truth with different label exists, LWL + LML

In the future, Reallabel will use a "**disagreement score**" by default (Aggregated Confidence - ground truth) where ground truth is 1 if it has the same label as the box group and 0 otherwise.  Disagreement scores near +1 suggest missed labels (LML), near -1 suggest wrong labels (LWL), and near 0 suggesting correct labels (LCL).

## Running RealLabel

### Set Up

The most common configuration options are:
- **deduplication_iou_threshold** – bounding boxes with Intersection over Union (IoU) below this threshold will be part of the same box group from which a single box winner will be calculated. Default: 0.5.
- **use_thresholds** – A boolean specifying whether or not to use the above thresholds to classify labels into discrete categories of Likely Correct/Wrong/Missed. Default: True, will be set to False in a future version.
    - The following config options are only used when use_thresholds is True:
        - **threshold_max_aggregated_confidence_fp** – Box group winners with AC Scores below this threshold will be classified as False Positive by RL (assuming ground truth in box group).  Box group winners with AC Scores above this threshold are classified as True Positive by RL (assuming ground truth is in box group). Default: 0.5.
        - **threshold_min_aggregated_confidence_fn** – Box group winners with AC Scores above this threshold will be classified as False Negative by RL (assuming no ground truth in box group).  Default: 0.5.
- **class_agnostic** – Set to True for when you only care about the LOCATIONS of bounding boxes (and not their labels). If True, RealLabel treats all classes as the same and there will be at most one output per de-duplication winner. Default: False.
- **run_confidence_calibration** – Whether RealLabel should calibrate the different models’ confidence scores. Default: False.
- **column_names** – A ColumnNameConfig object that defines the names of various columns in the data.
- **run_with_ground_truth** - If you have an unlabeled dataset and you'd like to run RealLabel as a first pass at labeling it, then set this to False.  RealLabel will classify detected objects as Likely Missing Labels (False Negatives).

## RealLabel Example

In [ ]:
import os
import time
from pathlib import Path
from typing import Any, Dict, Tuple

import numpy as np
import torch
from maite.protocols.object_detection import Dataset
from IPython.display import Image, Markdown

from reallabel import ColumnNameConfig, RealLabelConfig, RealLabelColumns
from checkmaite.object_detection.datasets import (
    CocoDetectionDataset,
    VisdroneDetectionDataset,
    YoloDetectionDataset,
)
from checkmaite.object_detection.models import TorchvisionODModel
from checkmaite.object_detection.test_stages import (
    RealLabelTestStage,
)

### 1. Create output directory for results

In [ ]:
output_dir = Path("reallabel_example_output")
os.makedirs(output_dir, exist_ok=True)

### 2. Load the dataset

In [ ]:
print("Loading COCO dataset...")
BASE_DIR = Path.cwd().parents[1]

# Use the example COCO dataset
root_path = BASE_DIR / "tests/testing_utilities/example_data/coco_dataset"
ann_file_path = BASE_DIR / "tests/testing_utilities/example_data/coco_dataset/ann_file.json"

dataset = CocoDetectionDataset(root=root_path, ann_file=ann_file_path, dataset_id="coco-example")
print(f"Dataset loaded with {len(dataset)} images")

### 3. Load models on the selected device

In [ ]:
models_config = {
    "fasterrcnn": "fasterrcnn_resnet50_fpn",
    "retinanet": "retinanet_resnet50_fpn",
}

models = {}
for model_name in models_config:
    models[model_name] = TorchvisionODModel(
        model_name=models_config[model_name],
        model_id=models_config[model_name],
    )
print(f"Loaded models: {', '.join(models_config.keys())}")

### 4. Configure RealLabel

In [ ]:
reallabel_config = RealLabelConfig(
    column_names={'unique_identifier_columns': ['id'], 'calibrated_confidence_column': 'score'},
    class_agnostic=True,
    additional_columns_clean_results=['aggregated_confidence'],
    run_with_ground_truth=True,
)


### 5. Create and configure RealLabel test stage

In [ ]:
test_stage = RealLabelTestStage()
test_stage.load_models(models=models)
test_stage.load_dataset(
    dataset=dataset,
    dataset_id="coco",
)

### 6. Run RealLabel

In [ ]:
%%time
run = test_stage.run(use_stage_cache=False, config=reallabel_config)

### 7. View Results

Detailed results are included in `run.outputs.results`

In [ ]:
run.outputs.results

In [ ]:
run.outputs.example_image.image

Running the cell above writes one of the images with RealLabel assignments to an output directory.
The bounding box colors have the following meaning:
- <span style="color:green">green</span> = likely correct label(true positive)
- <span style="color:blue">blue</span> = likely wrong label (false positive)
- <span style="color:red">red</span> = likely missed (false negative)

The aggregated confidence (AC) is shown in the top left of each box.

In [ ]:
# additional outputs are included that may be useful
print('Full List of Outputs:')
for key in run.outputs.model_fields.keys():
    print("- " + key)

# See RealLabel documentation for more info

### 8. Generate report

A powerpoint report and an html report with select results can be generated programmatically as shown below.

In [ ]:
from gradient.templates_and_layouts.create_deck import create_deck

slide_info = test_stage.collect_report_consumables()
powerpoint_report_path = create_deck(slide_info, output_dir, deck_name='RealLabel_Example_Report')

In [ ]:
display(Markdown(f"Presentation results are saved in the top level of `{output_dir}` directory."))

## Additonal Resources
Learn more about RealLabel with the following resources:
- [RealLabel Documentation](https://jatic.pages.jatic.net/morse/reallabel/)
- [RealLabel Config Documentation](https://jatic.pages.jatic.net/morse/reallabel/user_guide/reference/config_glossary.html#additional-outputs)
- [RealLabel Code Repository](https://gitlab.jatic.net/jatic/morse/reallabel)